In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from ultralytics import YOLO
import torch
print(torch.cuda.is_available())
torch.__version__

True


'2.11.0+cu128'

In [3]:
#model = YOLO('yolov8n.pt')
model = YOLO('./runs/detect/runs/train/vehicle_detection/weights/last.pt')
print(os.getcwd())

/home/windows11/llm-26/YOLO


In [ ]:
results = model.train(
    data='./trafic_data/data_1.yaml',  # 数据集配置文件
    epochs=100,            # 训练轮数
    batch=16,              # 批次大小
    imgsz=640,             # 输入图像尺寸
    device=0,
    workers=8,
    augment=True,          # 启用数据增强
    resume=True,          # 是否从断点恢复训练
    project='runs/train',  # 结果保存目录
    name='vehicle_detection', # 当前实验名称
    exist_ok=True
    )

New https://pypi.org/project/ultralytics/8.4.51 available 😃 Update with 'pip install -U ultralytics'
WARNING ⚠️ model 'runs/detect/runs/train/vehicle_detection/weights/last.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.49 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./trafic_data/data_1.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015

In [2]:
import cv2
model = YOLO('./runs/detect/runs/train/vehicle_detection/weights/best.pt')
cap = cv2.VideoCapture('video.mp4')
frame_num = 0

fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

line_start = (width // 2, 0)
line_end = (width // 2, height)
line_x = width // 2

fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('output_tracked.avi', fourcc, fps, (width, height))

counted_ids = set()      # 已计数的目标ID
total_count = 0          # 总越线数
last_positions = {}      # 记录上一帧每个目标的中心点 y 坐标
object_side = {}

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_num += 1
    results = model.track(frame, persist=True, tracker="bytetrack.yaml")

    cv2.line(frame, line_start, line_end, (0, 0, 255), 2)
    cv2.putText(frame, "Count Line", (line_start[0] + 10, line_start[1] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    
    if results[0].boxes.id is not None:
        # 获取检测框、置信度、类别和跟踪ID
        boxes = results[0].boxes.xyxy.cpu().numpy()  
        confs = results[0].boxes.conf.cpu().numpy()  
        class_ids = results[0].boxes.cls.cpu().numpy()  
        track_ids = results[0].boxes.id.cpu().numpy()  

        for box, conf, cls_id, track_id in zip(boxes, confs, class_ids, track_ids):
            x1, y1, x2, y2 = map(int, box)
            #中心点
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            class_name = model.names[cls_id]
            label = f"{class_name} #{track_id}"
            
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

            if cx < line_x:
                current_side = 'left'
            else:
                current_side = 'right'
            # 如果该目标已有记录
            if track_id in object_side:
                prev_side = object_side[track_id]
                if prev_side != current_side:
                    total_count += 1
                    # 更新侧边
                    object_side[track_id] = current_side
            else:
                # 首次出现，记录当前侧
                object_side[track_id] = current_side

    cv2.putText(frame, f"Total Crossings: {total_count}", (50, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    out.write(frame)
    
    cv2.imshow('YOLOv8 Tracking', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()

    



0: 384x640 2 cars, 2 motorbikes, 1 pickup, 1 suv, 25.3ms
Speed: 4.6ms preprocess, 25.3ms inference, 25.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 motorbikes, 1 pickup, 12.6ms
Speed: 2.1ms preprocess, 12.6ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 3 motorbikes, 1 pickup, 5.9ms
Speed: 1.5ms preprocess, 5.9ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 motorbikes, 1 pickup, 5.9ms
Speed: 1.7ms preprocess, 5.9ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 motorbikes, 1 pickup, 6.1ms
Speed: 1.1ms preprocess, 6.1ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 motorbikes, 1 suv, 6.3ms
Speed: 1.1ms preprocess, 6.3ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 motorbikes, 1 suv, 5.8ms
Speed: 1.2ms preprocess, 5.8ms inference, 1.4ms postprocess per image at s

QFontDatabase: Cannot find font directory /home/windows11/anaconda3/envs/llm-26-gpu/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/windows11/anaconda3/envs/llm-26-gpu/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/windows11/anaconda3/envs/llm-26-gpu/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/windows11/anaconda3/envs/llm-26-gpu/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconf

0: 384x640 1 car, 3 motorbikes, 1 suv, 6.5ms
Speed: 1.1ms preprocess, 6.5ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 motorbikes, 1 suv, 11.6ms
Speed: 1.1ms preprocess, 11.6ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 motorbikes, 1 suv, 5.7ms
Speed: 1.1ms preprocess, 5.7ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 motorbikes, 1 suv, 6.1ms
Speed: 1.4ms preprocess, 6.1ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 motorbikes, 1 suv, 5.4ms
Speed: 1.1ms preprocess, 5.4ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 motorbikes, 1 suv, 5.9ms
Speed: 1.0ms preprocess, 5.9ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 motorbikes, 1 suv, 6.4ms
Speed: 1.1ms preprocess, 6.4ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 3

In [16]:
type(frame)

NoneType